# Module 05 Lab - Data Preparation**Objective:** To learn and apply the most common data preparation techniques. Raw data is rarely ready for a machine learning model. This process, also called preprocessing, is one of the most critical steps in the entire ML workflow.**In this lab, you will write more of the code.** Read the explanations and then complete the tasks in the code cells.

## Part 1: Setup and Initial LookWe will continue using the Titanic dataset because it has the exact problems we need to solve: missing values and non-numeric data.

In [64]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Let's look at the missing values
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())


--- Missing Values Before Cleaning ---
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## Part 2: Handling Missing Values (Imputation)**Concept:** Most machine learning models cannot handle missing values (`NaN`). We must deal with them. Dropping the rows is an option, but you lose data. A better way is **imputation**, which means filling in the missing values with a calculated guess.Common imputation strategies:*   **Mean:** Fill with the average value. Good for normally distributed data.*   **Median:** Fill with the middle value. Better for skewed data or data with outliers (like `Fare`).*   **Mode:** Fill with the most frequent value. Used for categorical data.

### Task 1: Impute the 'Age' ColumnThe 'Age' column is missing many values. Since age can be skewed (e.g., by a few very old passengers), using the **median** is a robust choice.**Your Task:** Calculate the median of the 'Age' column and use the `.fillna()` method to replace the missing values.

In [65]:
# 1. Calculate the median of the 'Age' column
median_age = df['Age'].median()

# 2. Fill the missing values in 'Age' with the median
df['Age'] = df['Age'].fillna(median_age)

# 3. Verify that there are no more missing values in 'Age'
print("Missing values in 'Age' after imputation:")
print(df['Age'].isnull().sum())

Missing values in 'Age' after imputation:
0


## Part 3: Encoding Categorical Features**Concept:** Machine learning models are mathematical, so they need numbers, not text. We need to convert categorical columns (like 'Sex' and 'Embarked') into a numerical format. The most common method is **One-Hot Encoding**.One-Hot Encoding takes a column with `N` categories and turns it into `N` new columns, each with a `1` or `0`. For example, the 'Sex' column (`male`, `female`) becomes two new columns: `Sex_male` and `Sex_female`.Pandas has a convenient function called `pd.get_dummies()` that does this for us.

### Task 2: One-Hot Encode Categorical Columns**Your Task:** Use `pd.get_dummies()` to encode the 'Sex' and 'Embarked' columns. Make sure to drop the original columns after encoding.

In [66]:
# 1. Use get_dummies to create new columns for 'Sex' and 'Embarked'
#    Set `drop_first=True` to avoid multicollinearity (a statistical issue), which drops one of the new columns (e.g., just having `Sex_male` is enough to know if someone is female).
# encoded_df = pd.get_dummies(df, columns=[... , ...], drop_first=True)
encoded_df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# 2. Display the first few rows of the new DataFrame to see the new columns
print(encoded_df.head())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0      1      0   
2                             Heikkinen, Miss. Laina  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0      1      0   
4                           Allen, Mr. William Henry  35.0      0      0   

             Ticket     Fare Cabin  Sex_male  Embarked_Q  Embarked_S  
0         A/5 21171   7.2500   NaN      True       False        True  
1          PC 17599  71.2833   C85     False       False       False  
2  STON/O2. 3101282   7.9250   NaN     False       False        True  
3            113803  53.1000  C123     Fal

## Part 4: Feature Scaling**Concept:** Many models are sensitive to the scale of the features. For example, `Age` (from 0-80) and `Fare` (from 0-512) are on very different scales. This can cause the model to incorrectly believe that `Fare` is a more important feature simply because its values are larger.**Feature Scaling** solves this by putting all features on a similar scale. A common method is **Standardization** (`StandardScaler` in scikit-learn), which rescales the data to have a mean of 0 and a standard deviation of 1.**Important:** You only scale your numerical features, not your target variable or your newly encoded categorical columns.

### Task 3: Scale the 'Age' and 'Fare' Columns**Your Task:** Use `StandardScaler` from `sklearn.preprocessing` to scale the 'Age' and 'Fare' columns.

In [67]:
from sklearn.preprocessing import StandardScaler

# 1. Create an instance of the StandardScaler
scaler = StandardScaler()

# 2. Select the columns to scale
columns_to_scale = ['Age', 'Fare']

# 3. Fit the scaler to the data and transform it
#    Note: We are using the `encoded_df` from the previous step if you created it.
encoded_df[columns_to_scale] = scaler.fit_transform(encoded_df[columns_to_scale])

# 4. Display the first few rows to see the scaled data
print(encoded_df.head())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name       Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris -0.565736      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  0.663861      1      0   
2                             Heikkinen, Miss. Laina -0.258337      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  0.433312      1      0   
4                           Allen, Mr. William Henry  0.433312      0      0   

             Ticket      Fare Cabin  Sex_male  Embarked_Q  Embarked_S  
0         A/5 21171 -0.502445   NaN      True       False        True  
1          PC 17599  0.786845   C85     False       False       False  
2  STON/O2. 3101282 -0.488854   NaN     False       False        True  
3            1

## 📝 Knowledge Check

1. **Why is it often better to impute missing values with the median instead of the mean?**  

   **Alissa**: The median is often better than the mean because it is less affected by extreme values or outliers. For example, if a dataset contains a few unusually high or low values, the mean can become skewed and not represent the typical data point accurately. The median represents the middle value, so it provides a more robust estimate when the data distribution is uneven or contains outliers, which is common in real-world datasets like age or income.

   **Huong:** It is often better to use the median instead of the mean because the median is not affected by extreme values (outliers). The mean can be pulled higher or lower if there are unusually large or small values in the dataset. For example, if a few passengers are extremely old, the mean age would increase significantly, but the median would still represent the typical middle value. Since real-world data is often skewed, the median provides a more robust and stable estimate when filling in missing values.

   **Stuart:** The median can give the most common value in the data, where the mean can skew the dataset by giving the average where it may not make sense, e.g. 3 standard ticket prices averaged would give a fourth value, where choosing the most common value with the median would stay in the data shape of 3 values.

2. **Explain in your own words what One-Hot Encoding does and why it is necessary.**  

   **Alissa:** One-Hot Encoding converts categorical text data into numerical values that machine learning models can understand. It does this by creating new binary columns for each category, where a value of 1 indicates the presence of that category and 0 indicates its absence. This is necessary because most machine learning algorithms work with numbers, not text, and using raw categorical labels could incorrectly imply an order or relationship between categories that does not exist.

   **Huong:** One-Hot Encoding converts categorical text values into numerical columns that machine learning models can understand. Instead of storing categories like “male” and “female” as text, it creates new binary columns, where the value is 1 if true and 0 if false. This is necessary because most machine learning algorithms cannot process text directly. They require numerical input to perform mathematical calculations. One-Hot Encoding ensures that categorical variables are represented in a format the model can use without incorrectly assuming an order between categories.

   **Stuart:** Hot one turns a multi-value non-numerical column into many binary true/false columns in the dataset, making it computable in data while not modifying the dataset.

3. **Would you need to apply Feature Scaling to a Decision Tree model? Why or why not?**  

   **Alissa:** No, feature scaling is usually not necessary for a Decision Tree model. Decision Trees make decisions based on splitting data according to thresholds (for example, Age > 30), not on distances between values. Since scaling does not change the relative order of the data, it does not affect how the tree chooses its splits. Therefore, scaling is more important for models like k-nearest neighbors or neural networks, but not for Decision Trees.

   **Huong:** Usually no, Feature Scaling is not required for a Decision Tree model. Decision Trees make splits based on thresholds (greater than / less than) (for example, Age < 30), and they only care about the relative ordering of values, not their scale. Scaling changes the magnitude of numbers but does not change their order. Therefore, scaling does not affect how a Decision Tree decides where to split. However, scaling is important for models like K-Nearest Neighbors or Logistic Regression, which rely on distance calculations or gradient optimization.

   **Stuart:** It's not necessary to scale the features in a decision tree, as the decision tree will operate the same if it's scaled or not. e.g. With a 1-100 value, a >50 decision tree filter would give the same value even if the data is scaled to 0.01 to 1.00, as the filter is based on the order of values, not scale.
